In [49]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
DATA = Path("data")
DERIVED = Path("derived")

In [67]:
# importing data

arsenal26 = pd.read_csv(DATA / 'pitch_arsenals26.csv')
arsenal25 = pd.read_csv(DATA / 'pitch_arsenals25.csv')
arsenal24 = pd.read_csv(DATA / 'pitch_arsenals24.csv')
movement26 = pd.read_csv(DATA / 'pitch_movement26.csv')
movement25 = pd.read_csv(DATA / 'pitch_movement25.csv')
movement24 = pd.read_csv(DATA / 'pitch_movement24.csv')
stats26 = pd.read_csv(DATA / 'pitch-arsenal-stats126.csv')
stats25 = pd.read_csv(DATA / 'pitch-arsenal-stats125.csv')
stats24 = pd.read_csv(DATA / 'pitch-arsenal-stats124.csv')
qualifiers26 = pd.read_csv(DATA / 'pitch-arsenal-stats30026.csv')
qualifiers25 = pd.read_csv(DATA / 'pitch-arsenal-stats30025.csv')
qualifiers24 = pd.read_csv(DATA / 'pitch-arsenal-stats30024.csv')
arm_angle26 = pd.read_csv(DATA / 'pitcher_arm_angles26.csv')
arm_angle25 = pd.read_csv(DATA / 'pitcher_arm_angles25.csv')
arm_angle24 = pd.read_csv(DATA / 'pitcher_arm_angles24.csv')

In [74]:

merge02026 = pd.merge(arsenal26,arm_angle26,left_on='pitcher',right_on='pitcher',validate = 'one_to_one')
merge12026 = pd.merge(merge02026,movement26,left_on='pitcher', right_on='pitcher_id', validate='one_to_many')
full26 = pd.merge(merge12026, stats26, left_on=['pitcher','pitch_type'], right_on=['player_id','pitch_type'])

merge02025 = pd.merge(arsenal25,arm_angle25,left_on='pitcher',right_on='pitcher',validate = 'one_to_one')
merge12025 = pd.merge(merge02025,movement25,left_on='pitcher', right_on='pitcher_id', validate='one_to_many')
full25 = pd.merge(merge12025, stats25, left_on=['pitcher','pitch_type'], right_on=['player_id','pitch_type'])

merge02024 = pd.merge(arsenal24,arm_angle24,left_on='pitcher',right_on='pitcher',validate = 'one_to_one')
merge12024 = pd.merge(merge02024,movement24,left_on='pitcher', right_on='pitcher_id', validate='one_to_many')
full24 = pd.merge(merge12024, stats24, left_on=['pitcher','pitch_type'], right_on=['player_id','pitch_type'])

full26['year'] = 2026
full25['year'] = 2025
full24['year'] = 2024
full = pd.concat([full24,full25,full26],ignore_index = True)
full = full.drop(columns=['pitcher_name','team_name','last_name, first_name_y','pitch_hand_y','pitches','pitcher_id','n_pitches','pitch_type_name','last_name, first_name','player_id','team_name_alt','pitch_name'])



front = ['pitcher','last_name, first_name_x','team_name_abbrev', 'pitch_hand_x', 'year', 'pitch_type','pitches_thrown','pitch_usage','avg_speed','pitcher_break_x','pitcher_break_z_induced','run_value_per_100','run_value']
full = full[front + [c for c in full.columns if c not in front]]
full.rename(columns={"last_name, first_name_x": "name", "team_name_abbrev": "team", "pitcher":"id","pitch_hand_x":"hand","pitches_thrown":"n_pitches"}, inplace=True)

# Adds column on whether the pitch type ended 50 PA and was thrown at least 300 times
qualifiers24['year'] = 2024
qualifiers25['year'] = 2025
qualifiers26['year'] = 2026
qualifiers = pd.concat([qualifiers24, qualifiers25, qualifiers26], ignore_index=True)

qualifiers = qualifiers[['player_id', 'pitch_type', 'year']].rename(columns={'player_id': 'id'})
qualifiers['qualified'] = True

full = pd.merge(full, qualifiers, on=['id', 'pitch_type', 'year'], how='left', validate='one_to_one')
full['qualified'] = full['qualified'].fillna(False).astype(bool)

# Reorders the qualified column
front2 = ['id', 'name', 'team', 'hand', 'year', 'pitch_type', 'qualified', 'n_pitches', 'pitch_usage', 'avg_speed', 'pitcher_break_x', 'pitcher_break_z_induced', 'run_value_per_100', 'run_value']
full = full[front2 + [c for c in full.columns if c not in front2]]
full = full.rename(columns={"qualified": "pitch_type_qualified"})

In [75]:
full

,id,name,team,hand,year,pitch_type,pitch_type_qualified,n_pitches,pitch_usage,avg_speed,pitcher_break_x,pitcher_break_z_induced,run_value_per_100,run_value,n_ff,n_si,n_fc,n_sl,n_ch,n_cu,n_fs,n_kn,n_st,n_sv,team_id,ball_angle,relative_release_ball_x,release_ball_z,relative_shoulder_x,shoulder_z,total_pitches,pitches_per_game,pitch_per,pitcher_break_z,league_break_z,diff_z,rise,league_break_x,diff_x,tail,percent_rank_diff_z,percent_rank_diff_x,pa,ba,slg,woba,whiff_percent,k_percent,put_away,est_ba,est_slg,est_woba,hard_hit_percent
0,657277,"Webb, Logan",SF,R,2024,SI,True,1270,39.7,92.6,15.1,1.1,1.4,17,5.5,39.7,2.6,NaN,30.9,NaN,NaN,NaN,21.3,NaN,137,22.1,-2.664236,4.978202,-0.514977,4.103699,3200,19.753086,0.396875,30.9,-25.4,5.5,22.0,-15.5,-0.3,-2.0,0.968858,0.346021,315,0.268,0.338,0.306,14.1,16.5,20.5,0.259,0.375,0.317,44.7
1,657277,"Webb, Logan",SF,R,2024,CH,True,990,30.9,87.4,9.8,-4.8,0.1,1,5.5,39.7,2.6,NaN,30.9,NaN,NaN,NaN,21.3,NaN,137,22.1,-2.664236,4.978202,-0.514977,4.103699,3200,19.753086,0.309375,40.4,-34.9,5.6,16.0,-14.0,-4.1,-30.0,0.977099,0.030534,319,0.275,0.411,0.322,21.6,17.9,16.4,0.305,0.513,0.372,51.6
2,657277,"Webb, Logan",SF,R,2024,ST,True,680,21.3,83.8,16.4,-0.9,0.4,3,5.5,39.7,2.6,NaN,30.9,NaN,NaN,NaN,21.3,NaN,137,22.1,-2.664236,4.978202,-0.514977,4.103699,3200,19.753086,0.212500,39.9,-37.1,2.8,8.0,13.6,2.8,20.0,0.811321,0.855346,147,0.239,0.331,0.271,26.8,27.2,14.9,0.249,0.376,0.283,38.8
3,657277,"Webb, Logan",SF,R,2024,FF,False,176,5.5,92.6,8.6,9.2,-0.5,-1,5.5,39.7,2.6,NaN,30.9,NaN,NaN,NaN,21.3,NaN,137,22.1,-2.664236,4.978202,-0.514977,4.103699,3200,19.753086,0.055000,22.5,-17.6,-4.9,-28.0,-9.5,-0.9,-10.0,0.021583,0.352518,42,0.179,0.231,0.217,34.7,47.6,16.5,0.211,0.304,0.255,47.4
4,605400,"Nola, Aaron",PHI,R,2024,CU,True,1049,32.9,79.1,14.0,-9.9,1.2,12,27.7,20.6,9.2,NaN,9.6,32.9,NaN,NaN,NaN,NaN,143,20.1,-2.401195,5.024550,-0.244311,4.236191,3189,19.685185,0.328943,53.9,-52.4,1.5,3.0,12.2,1.9,15.0,0.557692,0.701923,304,0.221,0.381,0.285,37.0,31.3,22.3,0.216,0.351,0.268,33.3
5,605400,"Nola, Aaron",PHI,R,2024,FF,True,883,27.7,92.5,12.0,14.7,1.0,9,27.7,20.6,9.2,NaN,9.6,32.9,NaN,NaN,NaN,NaN,143,20.1,-2.401195,5.024550,-0.244311,4.236191,3189,19.685185,0.276889,17.4,-17.8,0.4,2.0,-9.6,2.4,25.0,0.628297,0.788969,201,0.167,0.316,0.269,23.0,24.9,18.4,0.187,0.331,0.285,39.2
6,605400,"Nola, Aaron",PHI,R,2024,SI,True,656,20.6,91.5,16.9,8.6,0.0,0,27.7,20.6,9.2,NaN,9.6,32.9,NaN,NaN,NaN,NaN,143,20.1,-2.401195,5.024550,-0.244311,4.236191,3189,19.685185,0.205707,24.1,-26.1,-2.0,-8.0,-15.4,1.5,10.0,0.276817,0.837370,188,0.315,0.478,0.357,12.8,20.2,22.9,0.274,0.443,0.327,44.7
7,605400,"Nola, Aaron",PHI,R,2024,CH,True,307,9.6,85.7,15.3,3.2,-1.1,-3,27.7,20.6,9.2,NaN,9.6,32.9,NaN,NaN,NaN,NaN,143,20.1,-2.401195,5.024550,-0.244311,4.236191,3189,19.685185,0.096268,34.0,-35.6,-1.7,-5.0,-14.5,0.8,5.0,0.297710,0.595420,71,0.279,0.426,0.321,13.5,8.5,8.8,0.341,0.499,0.383,38.7
8,605400,"Nola, Aaron",PHI,R,2024,FC,False,294,9.2,87.5,1.3,8.5,-2.6,-8,27.7,20.6,9.2,NaN,9.6,32.9,NaN,NaN,NaN,NaN,143,20.1,-2.401195,5.024550,-0.244311,4.236191,3189,19.685185,0.092192,27.1,-27.8,0.7,2.0,1.7,-2.9,-176.0,0.662577,0.104294,55,0.389,0.796,0.501,17.8,12.7,12.3,0.338,0.599,0.406,36.2
9,656302,"Cease, Dylan",SD,R,2024,FF,True,1386,43.5,96.9,2.5,18.4,0.2,3,43.5,0.1,0.6,43.0,0.9,7.9,NaN,NaN,4.1,NaN,135,53.4,-1.292627,6.100292,0.020948,4.327711,3188,19.679012,0.434755,10.8,-13.0,2.2,17.0,-7.4,-5.0,-67.0,0.937650,0.057554,324,0.232,0.387,0.342,20.3,19.8,17.2,0.241,0.445,0.358,44.0


Checks:

In [76]:
len(full) == len(full24) + len(full25) + len(full26)
full['year'].value_counts()

year
2026    1249
2025    1239
2024    1189
Name: count, dtype: int64

In [77]:
full.duplicated(subset=['id', 'pitch_type', 'year']).sum()

np.int64(0)

In [78]:
full.groupby('id')['year'].nunique().value_counts()

year
1    236
2    148
3    105
Name: count, dtype: int64

In [79]:
#data includes all pitch types used more than 100 times, but should have more than 25 PA to be used as 
full.to_csv(DERIVED / 'data24-26.csv',index=False)